# Market Index Service

Fetch **SET market index** data — the index directory (SET, SET50, SET50FF, SET100, SET100FF,
sSET, SETCLMV, SETHD, SETESG, SETWB, mai + industries + sectors), per-index quotations,
constituents with full quote rows, intraday chart series, and **index membership per stock**.

Endpoints:
- `GET /api/set/index/list` — index directory
- `GET /api/set/index/info/list?type=INDEX|INDUSTRY` — all quotes in one call
- `GET /api/set/index/{symbol}/info` — one index quotation (page-header data)
- `GET /api/set/index/{symbol}/composition` — constituents ("หลักทรัพย์ที่ใช้คำนวณดัชนี")
- `GET /api/set/index/{symbol}/chart-quotation` — intraday/historical series

> In Jupyter you can `await` directly — no `asyncio.run()` needed.

## Setup

```bash
pip install settfex pandas
```

In [ ]:
import pandas as pd

from settfex import SetIndex
from settfex.services.set import (
    get_index_composition,
    get_index_info,
    get_index_info_list,
    get_index_list,
    get_stock_list,
)

## 1. List All Market Indices

The directory has three levels: `INDEX` (the 11 headline indices), `INDUSTRY`, and `SECTOR`.
SET and mai share industry symbols — the mai variants are distinguished by a `-m` suffixed
`query_symbol` (e.g. `AGRO-m`).

In [ ]:
index_list = await get_index_list()

print(f"Total indices: {index_list.count}")
print(f"Headline:      {[ix.symbol for ix in index_list.market_indices]}")
print(f"Industries:    {len(index_list.industries)} | Sectors: {len(index_list.sectors)}")

In [ ]:
# Full directory as a DataFrame
df_indices = pd.DataFrame([ix.model_dump() for ix in index_list.indices])
df_indices.groupby(["level", "market"]).size()

In [ ]:
# SET vs mai industry disambiguation
set_agro = index_list.get_index("AGRO", market="SET")
mai_agro = index_list.get_index("AGRO-m")
print(f"SET AGRO -> query_symbol={set_agro.query_symbol!r}")
print(f"mai AGRO -> query_symbol={mai_agro.query_symbol!r} (market={mai_agro.market})")

## 2. Index Quotation (Page-Header Data)

Everything the set.or.th index page header shows: last value, change, %change,
open/high/low, volume, value, market status, and the data timestamp (tz-aware, Asia/Bangkok).

In [ ]:
index = SetIndex("SET50")
info = await index.get_info()

print(f"{info.symbol}")
print(f"  Last:    {info.last:,.2f}  ({info.change:+,.2f} / {info.percent_change:+.2f}%)")
print(f"  Open:    {info.open:,.2f}   High: {info.high:,.2f}   Low: {info.low:,.2f}")
print(f"  Volume:  {info.volume:,.0f} shares")
print(f"  Value:   {info.value:,.0f} THB")
print(f"  Status:  {info.market_status}")
print(f"  As of:   {info.market_date_time}")

In [ ]:
# Index symbols keep their casing — and the API resolves paths case-insensitively:
sset = await get_index_info("sset")
print(f"'sset' resolved to {sset.symbol!r}: last={sset.last}")

## 3. All Index Quotations in One Call

`type='INDEX'` returns the 11 headline indices; `type='INDUSTRY'` returns every industry
AND sector index for both markets.

In [ ]:
quotes = await get_index_info_list()

df_quotes = pd.DataFrame([q.model_dump() for q in quotes])
df_quotes[["symbol", "last", "change", "percent_change", "volume", "value", "market_status"]]

In [ ]:
industry_quotes = await get_index_info_list(index_type="INDUSTRY")
print(f"{len(industry_quotes)} industry/sector quotes")
pd.DataFrame([q.model_dump() for q in industry_quotes])[
    ["symbol", "market_name", "level", "last", "percent_change"]
].head(10)

## 4. Index Constituents

The securities used to calculate the index, each with a full quote row — the
"หลักทรัพย์ที่ใช้คำนวณดัชนี" table: OHLC, change, best bid/offer, volume, value,
plus market cap, P/E, P/B, dividend yield, and the 52-week range.

In [ ]:
composition = await index.get_composition()  # SetIndex("SET50")

print(f"{composition.composition.symbol}: {composition.count} constituents")
print(f"Index quote embedded: last={composition.index_info.last}")

rows = [
    {
        "symbol": c.symbol,
        "open": c.open,
        "high": c.high,
        "low": c.low,
        "last": c.last,
        "chg%": c.percent_change,
        "bid": c.best_bid,
        "offer": c.best_offer,
        "volume": c.total_volume,
        "value_kTHB": (c.total_value or 0) / 1e3,
        "mktcap_MB": (c.market_cap or 0) / 1e6,
        "P/E": c.pe_ratio,
    }
    for c in composition.constituents
]
df_members = pd.DataFrame(rows).sort_values("mktcap_MB", ascending=False)
df_members.head(10)

In [ ]:
# One-line convenience + a specific constituent
esg = await get_index_composition("SETESG")
print(f"SETESG members: {esg.count}")
cpall = esg.get_constituent("cpall")  # case-insensitive
print(f"CPALL in SETESG: last={cpall.last}, dividend_yield={cpall.dividend_yield}%")

## 5. Industry Drilldown (SET) vs Direct Stocks (mai)

SET industry indices (e.g. `AGRO`) return **no direct constituents** — they return their
sector quotes in `sub_indices` instead. mai industries (query symbol `AGRO-m`) return stocks
directly. The whole-market indices `SET`/`mai` have **no composition endpoint** (the service
raises a helpful error).

In [ ]:
agro = await get_index_composition("AGRO")  # SET industry -> sector drilldown
print(f"AGRO stocks: {agro.count}, sectors: {[s.symbol for s in agro.composition.sub_indices]}")

agro_m = await get_index_composition("AGRO-m")  # mai industry -> stocks
print(f"AGRO-m (mai) stocks: {agro_m.count} -> {agro_m.symbols[:5]} ...")

## 6. Intraday Chart & Latest Traded Index Value

The index chart-quotation shares the stock chart models, including the latest-*traded*-value
logic that skips the pre-populated future / lunch-break / no-trade buckets.

In [ ]:
chart = await index.get_chart_quotation(period="1D")
print(f"Prior close: {chart.prior} | points: {len(chart.quotations)}")

latest = await index.get_latest_price()
if latest:
    print(
        f"Latest traded: {latest.price:,.2f} @ {latest.local_datetime} ({latest.percent_change:+.2f}%)"
    )
else:
    print(f"Nothing traded yet today — prior close {chart.prior}")

## 7. Index Membership per Stock (Enriched Stock List)

`get_stock_list()` now joins the nine headline sub-index compositions into each stock **by
default** (~10 extra concurrent requests). Every `StockSymbol` gets an `indices` list, and
the response gains `filter_by_index()`. Pass `include_indices=False` for the previous
single-request behavior.

In [ ]:
stock_list = await get_stock_list()  # enrichment on by default

cpall = stock_list.get_symbol("CPALL")
print(f"CPALL indices: {cpall.indices}")

print(f"SET50 members:  {len(stock_list.filter_by_index('SET50'))}")
print(f"SETESG members: {len(stock_list.filter_by_index('setesg'))}   # case-insensitive")

In [ ]:
# Universe with memberships as a DataFrame (e.g. for a rebalance monitor)
df_universe = pd.DataFrame(
    [
        {
            "symbol": s.symbol,
            "market": s.market,
            "industry": s.industry,
            "indices": ", ".join(s.indices),
        }
        for s in stock_list.security_symbols
        if s.indices
    ]
)
print(f"{len(df_universe)} stocks belong to at least one headline sub-index")
df_universe.head(10)

In [ ]:
# Skip the enrichment when you only need the raw universe (fastest, single request)
plain = await get_stock_list(include_indices=False)
print(f"{plain.count} stocks, indices empty: {plain.security_symbols[0].indices == []}")

## Next Steps

- Build a **SET50/SET100 tracking universe** and monitor membership changes over time
- Screen the **SETESG**/**SETHD** constituent quote tables for strategy candidates
- Combine with the [Chart Quotation notebook](13_chart_quotation.ipynb) for per-stock intraday data
- See the full service docs: `docs/settfex/services/set/index.md`